In [1]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import warnings
warnings.filterwarnings('ignore')

print("All imports successful!")

All imports successful!


In [3]:
# Load all tickers
with open('../data/processed/ticker_list.txt', 'r') as f:
    tickers = [line.strip() for line in f.readlines()]

# Load all data
dfs = []
for ticker in tickers:
    try:
        df = pd.read_csv(f'../data/processed/{ticker}_features.csv')
        dfs.append(df)
    except:
        pass

data = pd.concat(dfs, ignore_index=True)
data['Date'] = pd.to_datetime(data['Date'])
data = data[data['Date'] >= pd.Timestamp('2024-01-01')]

# Feature columns
drop_cols = ['Date', 'Open', 'High', 'Low', 'Close', 'Volume',
             'Trades', 'Turnover', 'Ticker', 'Label', 'Future_Return', 'Sector']
feature_cols = [c for c in data.columns if c not in drop_cols]

print(f"Total rows: {len(data)}")
print(f"Features: {len(feature_cols)}")
print(f"Sectors: {data['Sector'].nunique()}")

Total rows: 139651
Features: 23
Sectors: 12


In [5]:
# ── LSTM Sequence Creator ─────────────────────────────────────
SEQ_LEN = 10

def create_sequences(X, y, seq_len):
    Xs, ys = [], []
    for i in range(len(X) - seq_len):
        Xs.append(X[i:i+seq_len])
        ys.append(y[i+seq_len])
    return np.array(Xs), np.array(ys)

# ── LSTM Model ────────────────────────────────────────────────
class StockLSTM(nn.Module):
    def __init__(self, input_dim, hidden_dim=128, num_layers=2, dropout=0.3):
        super(StockLSTM, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 3)
        )

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_out = lstm_out[:, -1, :]
        return self.classifier(last_out)

# ── Train function ────────────────────────────────────────────
def train_lstm(X_seq, y_seq, input_dim):
    X_train, X_test, y_train, y_test = train_test_split(
        X_seq, y_seq, test_size=0.2, random_state=42, stratify=y_seq
    )

    X_train_t = torch.FloatTensor(X_train)
    y_train_t = torch.LongTensor(y_train)
    X_test_t  = torch.FloatTensor(X_test)

    train_dl = DataLoader(
        TensorDataset(X_train_t, y_train_t),
        batch_size=128, shuffle=True
    )

    model = StockLSTM(input_dim=input_dim)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100)

    # Early stopping
    patience = 30
    best_acc = 0
    best_weights = None
    no_improve = 0

    for epoch in range(1000):
        model.train()
        for X_batch, y_batch in train_dl:
            optimizer.zero_grad()
            loss = criterion(model(X_batch), y_batch)
            loss.backward()
            optimizer.step()
        scheduler.step()

        model.eval()
        with torch.no_grad():
            _, predicted = torch.max(model(X_test_t), 1)
            acc = accuracy_score(y_test, predicted.numpy())

        if acc > best_acc:
            best_acc = acc
            best_weights = {k: v.clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= patience:
            break

    model.load_state_dict(best_weights)
    model.eval()
    with torch.no_grad():
        _, predicted = torch.max(model(X_test_t), 1)
        preds = predicted.numpy()

    return best_acc, preds, y_test

print("LSTM functions ready!")

LSTM functions ready!


In [6]:
target_sectors = [
    'Hydro Power',
    'Microfinance',
    'Commercial Banks',
    'Development Banks',
    'Finance',
    'Life Insurance',
    'Manufacturing And Processing',
    'Non Life Insurance',
]

results_lstm = {}
scaler = StandardScaler()

for sector in target_sectors:
    print(f"\n{'='*45}")
    print(f"Training: {sector}")

    # Filter sector data
    sector_data = data[data['Sector'] == sector].copy()
    sector_data = sector_data.dropna(subset=feature_cols + ['Label'])

    n_stocks = sector_data['Ticker'].nunique()
    n_rows = len(sector_data)
    print(f"Stocks: {n_stocks} | Rows: {n_rows}")

    if n_rows < 500:
        print(f"Skipping — not enough data")
        continue

    X = sector_data[feature_cols].values
    y = sector_data['Label'].values

    # Scale
    X_scaled = scaler.fit_transform(X)

    # Create sequences
    X_seq, y_seq = create_sequences(X_scaled, y, SEQ_LEN)
    print(f"Sequences: {X_seq.shape}")

    # Train LSTM
    best_acc, preds, y_test = train_lstm(X_seq, y_seq, X_seq.shape[2])

    print(f"Accuracy: {best_acc*100:.1f}%")
    print(classification_report(y_test, preds, target_names=['Sell', 'Hold', 'Buy']))

    results_lstm[sector] = {
        'accuracy': round(best_acc, 4),
        'stocks': n_stocks,
        'rows': n_rows
    }

# ── Summary ──────────────────────────────────────────────────
print("\n" + "="*55)
print("LSTM SECTOR SUMMARY")
print("="*55)
for sector, res in results_lstm.items():
    print(f"{sector:35} | Acc: {res['accuracy']*100:.1f}% | Stocks: {res['stocks']}")


Training: Hydro Power
Stocks: 97 | Rows: 52088
Sequences: (52078, 10, 23)
Accuracy: 48.6%
              precision    recall  f1-score   support

        Sell       0.48      0.41      0.44      3314
        Hold       0.51      0.67      0.58      4268
         Buy       0.43      0.31      0.36      2834

    accuracy                           0.49     10416
   macro avg       0.47      0.46      0.46     10416
weighted avg       0.48      0.49      0.47     10416


Training: Microfinance
Stocks: 50 | Rows: 26978
Sequences: (26968, 10, 23)
Accuracy: 51.6%
              precision    recall  f1-score   support

        Sell       0.47      0.29      0.36      1540
        Hold       0.54      0.79      0.64      2497
         Buy       0.47      0.28      0.35      1357

    accuracy                           0.52      5394
   macro avg       0.49      0.45      0.45      5394
weighted avg       0.50      0.52      0.49      5394


Training: Commercial Banks
Stocks: 19 | Rows: 10792
Se